In [ ]:
def photostim_prepost_frames(epoch_table):
    # This will pull the photostim and photostim_post stimuli only and add a new column (duration)
    relevant_epochs = epoch_table[epoch_table['stimulus_name'].isin(photo_test['stimulus_name'])].copy()
    relevant_epochs['duration'] = relevant_epochs.stop_time-relevant_epochs.start_time
    
    # Now get the start and stop times for the photostim epochs before and after the BCI task

    photostim_pre_start = relevant_epochs.loc[1].start_frame
    photostim_pre_stop = relevant_epochs.loc[1].stop_frame

    photostim_post_start = relevant_epochs.loc[4].start_frame
    photostim_post_stop = relevant_epochs.loc[4].stop_frame
    
    return relevant_epochs, photostim_pre_start, photostim_pre_stop, photostim_post_start, photostim_post_start

In [ ]:
def correlation_prepost_bci(dff_traces, photostim_pre_start, photostim_pre_stop, 
                           photostim_post_start, photostim_post_stop, valid_roi_table):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import stats
    
    # Index into the dff traces array to limit to the relevant time points
    dff_pre = dff_traces[photostim_pre_start:photostim_pre_stop, :]
    dff_pre = dff_pre.T
    dff_post = dff_traces[photostim_post_start:photostim_post_stop, :]
    dff_post = dff_post.T
                            
    # Limit dff traces for photostimulation activity period before the task to valid ROIs
    dff_pre = dff_pre[valid_roi_table.index.values, :]
    print(f"dff_pre shape: {dff_pre.shape}")
    
    # Limit dff traces for photostimulation activity period after the task to valid ROIs
    dff_post = dff_post[valid_roi_table.index.values, :]
    print(f"dff_post shape: {dff_post.shape}")
    
    # Calculate correlation matrices
    correlations_pre = np.corrcoef(dff_pre)
    correlations_post = np.corrcoef(dff_post)
    
    # Keep only upper triangular part (avoid duplicate pairs and diagonal)
    correlations_pre = np.triu(correlations_pre, k=1)
    correlations_post = np.triu(correlations_post, k=1)
    
    # Statistical comparison
    t, p_val = stats.ttest_ind(correlations_pre.flatten(), correlations_post.flatten())
    print(f"T-test p-value: {p_val}")
    
    # Plot both distributions
    plt.figure(figsize=(10, 6))
    plt.hist(correlations_pre.flatten(), bins=100, color='green', label='pre', histtype='step')
    plt.hist(correlations_post.flatten(), bins=100, color='purple', label='post', histtype='step')
    plt.title('Correlations before and after BCI learning, p=' + str(np.round(p_val, 5)))
    plt.xlabel('Correlation')
    plt.ylabel('# cell pairs')
    plt.xlim(-0.1, 0.1)
    plt.legend()
    plt.show()
    
    return correlations_pre, correlations_post, p_val

In [ ]:
def get_trial_correlations(trial_list, dff_traces):
    """Calculate correlation matrix for a set of trials"""
    all_trial_data = []
    
    for _, trial in trial_list.iterrows():
        start_frame = trial['start_frame']
        stop_frame = trial['stop_frame']
        trial_dff = dff_traces[start_frame:stop_frame, :].T  # Transpose so neurons are rows
        all_trial_data.append(trial_dff)
    
    # Concatenate all trials along time dimension
    combined_data = np.concatenate(all_trial_data, axis=1)
    
    # Calculate correlation matrix (neurons x neurons)
    correlation_matrix = np.corrcoef(combined_data)
    
    return correlation_matrix
